# LeetCode 3725 — Count Ways to Choose Coprime Integers from Rows bằng Python

- Link: https://leetcode.com/problems/count-ways-to-choose-coprime-integers-from-rows/
- Độ khó: **Hard**
- Chủ đề: Số học, **DP theo ước chung lớn nhất**, **hàm Möbius**, bao hàm – loại trừ trên ước, đếm modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countCoprime(self, mat: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. DP theo trạng thái "gcd của các số đã chọn": chỉ có tối đa 150 giá trị.
2. Đếm "mọi số chọn đều chia hết cho `d`" dễ dàng: `f(d) = Π_hàng #{x trong hàng : d | x}`.
3. Từ `f(d)` suy ra số bộ có gcd **đúng bằng** `d` bằng trừ dần từ lớn xuống nhỏ, hoặc bằng **công thức Möbius** `Σ_d μ(d)·f(d)`.
4. Tính hàm Möbius bằng sàng.
5. Kiểm chứng bằng vét cạn tích Descartes.

## 2. Đề bài & ví dụ

Cho ma trận `m × n` số nguyên dương. Đếm số cách chọn **đúng một số mỗi hàng** sao cho gcd của các số được chọn bằng `1`, modulo `10^9 + 7`. (Các ô khác nhau là các lựa chọn khác nhau, kể cả khi cùng giá trị.)

**Ví dụ 1:** `[[1,2],[3,4]]` → `3`.

**Ví dụ 2:** `[[2,2],[2,2]]` → `0`.

**Ràng buộc:** `m, n ≤ 150`, `1 ≤ mat[i][j] ≤ 150`.

In [1]:
from typing import List
from math import gcd
from itertools import product
from functools import reduce
import random, time

MOD = 10**9 + 7
examples = [([[1, 2], [3, 4]], 3), ([[2, 2], [2, 2]], 0)]
for mat, exp in examples:
    print(mat, "-> kỳ vọng", exp)

[[1, 2], [3, 4]] -> kỳ vọng 3
[[2, 2], [2, 2]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. gcd của nhiều số

`gcd(a, b, c) = gcd(gcd(a, b), c)` ⇒ gộp dần từng hàng. gcd luôn là ước của số đầu tiên ⇒ trạng thái nằm trong `[1, 150]`.

In [2]:
print(reduce(gcd, [12, 18, 30]), reduce(gcd, [7, 14, 15]))

6 1


### 3.2. Đếm bộ mà mọi phần tử chia hết cho `d`

Các hàng độc lập ⇒ `f(d) = Π_i c_i(d)` với `c_i(d)` = số phần tử hàng `i` chia hết cho `d`. Một bộ có `d | gcd` ⇔ mọi phần tử chia hết cho `d`.

### 3.3. Hàm Möbius

`μ(1) = 1`; `μ(d) = (−1)^r` nếu `d` là tích của `r` số nguyên tố **phân biệt**; `μ(d) = 0` nếu `d` có ước chính phương `> 1`. Tính chất then chốt: `Σ_{d | n} μ(d) = [n = 1]`.

In [3]:
def mobius_sieve(N):
    mu = [1] * (N + 1); is_comp = [False] * (N + 1); primes = []
    mu[0] = 0
    for i in range(2, N + 1):
        if not is_comp[i]:
            primes.append(i); mu[i] = -1
        for p in primes:
            if i * p > N: break
            is_comp[i * p] = True
            if i % p == 0:
                mu[i * p] = 0; break
            mu[i * p] = -mu[i]
    return mu

mu = mobius_sieve(150)
print("μ(1..12) =", mu[1:13])
assert all(sum(mu[d] for d in range(1, n + 1) if n % d == 0) == (1 if n == 1 else 0) for n in range(1, 151))

μ(1..12) = [1, -1, -1, 0, -1, 1, -1, 0, 0, 1, -1, 0]


### 3.4. Vì sao `Σ_{d | n} μ(d) = [n = 1]`?

Nếu `n = p₁^{a₁}⋯p_r^{a_r}` với `r ≥ 1`, các ước `d` có `μ(d) ≠ 0` là tích của một **tập con** các số nguyên tố `{p₁, …, p_r}`. Tổng `Σ_{S ⊆ [r]} (−1)^{|S|} = (1 − 1)^r = 0`. Với `n = 1` chỉ có `d = 1` ⇒ tổng bằng 1. Đây chính là **bao hàm – loại trừ** trên tập các số nguyên tố chia `gcd`.

In [4]:
n = 60
divs = [d for d in range(1, n + 1) if n % d == 0]
print("ước của 60:", divs)
print("μ tương ứng:", [mu[d] for d in divs], "| tổng =", sum(mu[d] for d in divs))

ước của 60: [1, 2, 3, 4, 5, 6, 10, 12, 15, 20, 30, 60]
μ tương ứng: [1, -1, -1, 0, -1, 1, 1, 0, 1, 0, -1, 0] | tổng = 0


### 3.5. Đếm bội trong `O(V log V)`

Với bảng tần suất `cnt[x]`, số phần tử chia hết cho `d` là `cnt[d] + cnt[2d] + …`. Tổng số phép cộng trên mọi `d` là `V/1 + V/2 + … + V/V ≈ V ln V` — chuỗi điều hoà. Với `V = 150` chỉ khoảng 800 phép cộng mỗi hàng.

In [5]:
print("số phép cộng khi đếm bội, V = 150:", sum(150 // d for d in range(1, 151)))

số phép cộng khi đếm bội, V = 150: 780


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Số bộ có gcd = 1 = `Σ_{bộ} [gcd = 1]` = `Σ_{bộ} Σ_{d | gcd} μ(d)` = `Σ_d μ(d) · #{bộ : d | gcd}` = `Σ_d μ(d) · f(d)`.

Ví dụ 1: `f(1) = 2·2 = 4`, `f(2) = 1·1 = 1` (chọn 2 và 4), `f(3) = 0·1 = 0`, … ⇒ `μ(1)·4 + μ(2)·1 = 4 − 1 = 3`. ✔

## 5. Cách 1 — Brute Force: duyệt tích Descartes

In [6]:
def brute(mat):
    return sum(1 for pick in product(*mat) if reduce(gcd, pick) == 1) % MOD

for mat, exp in examples:
    print(brute(mat), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0


**Nút thắt:** `n^m` bộ — với `150^150` là không tưởng.

## 6. Ý tưởng thuật toán / bước đột phá

Hai hướng đều dựa vào việc giá trị chỉ tới `V = 150`:

- **DP theo gcd:** `dp[g]` = số cách chọn các hàng đã xét có gcd `= g`. Gộp một hàng bằng bảng tần suất giá trị của hàng: `new[gcd(g, x)] += dp[g] · cnt[x]` ⇒ `O(m · V²)`.
- **Möbius:** tính `c_i(d)` cho mọi hàng và mọi `d` (đếm bội) ⇒ `f(d)` ⇒ `Σ μ(d) f(d)` ⇒ `O(m·(n + V log V))`.

**So sánh trực giác.** DP gcd "đi theo hàng" và theo dõi **chính xác** gcd hiện tại — tự nhiên nhưng mỗi hàng tốn `V × (số giá trị khác nhau)`. Möbius "đi theo ước": thay vì hỏi "gcd bằng bao nhiêu", ta hỏi câu dễ hơn "gcd có chia hết cho `d` không" (tích độc lập theo hàng), rồi dùng nghịch đảo Möbius để chuyển từ "chia hết cho" sang "bằng". Đây là mẫu chung: **đếm theo điều kiện chia hết dễ hơn đếm theo đẳng thức**.

Lưu ý thêm: chỉ cần `d ≤ max(mat)`, vì `f(d) = 0` khi `d` lớn hơn mọi phần tử.

## 7. Các cách tiếp theo

- **Cách 2:** DP theo gcd — `O(m · V²)`.
- **Cách 3:** "gcd đúng bằng `d`" bằng trừ dần: `h(d) = f(d) − Σ_{k ≥ 2} h(k·d)`, duyệt `d` từ `V` xuống `1` — `O(m·n + V log V)`.
- **Cách 4:** công thức Möbius `Σ μ(d) f(d)` (lời giải nộp).

### 7.1. Cách 2 — DP theo gcd

In [7]:
V = 150

def dp_gcd(mat):
    dp = [0] * (V + 1)
    for x in mat[0]: dp[x] += 1
    for row in mat[1:]:
        cnt = [0] * (V + 1)
        for x in row: cnt[x] += 1
        vals = [x for x in range(1, V + 1) if cnt[x]]
        new = [0] * (V + 1)
        for g in range(1, V + 1):
            if dp[g]:
                for x in vals:
                    h = gcd(g, x); new[h] = (new[h] + dp[g] * cnt[x]) % MOD
        dp = new
    return dp[1] % MOD

for mat, exp in examples:
    print(dp_gcd(mat), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0


### 7.2. Cách 3 — Trừ dần từ ước lớn

In [8]:
def multiples_count(row):
    cnt = [0] * (V + 1)
    for x in row: cnt[x] += 1
    c = [0] * (V + 1)
    for d in range(1, V + 1):
        c[d] = sum(cnt[k] for k in range(d, V + 1, d))      # số phần tử chia hết cho d
    return c

def f_all(mat):
    f = [1] * (V + 1); f[0] = 0
    for row in mat:
        c = multiples_count(row)
        for d in range(1, V + 1):
            f[d] = f[d] * c[d] % MOD
    return f

def exact_by_subtraction(mat):
    f = f_all(mat); h = [0] * (V + 1)
    for d in range(V, 0, -1):
        h[d] = f[d]
        for k in range(2 * d, V + 1, d):
            h[d] -= h[k]
        h[d] %= MOD
    return h[1]

for mat, exp in examples:
    print(exact_by_subtraction(mat), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0


### 7.3. Cách 4 — Möbius (lời giải chính)

In [9]:
def mobius_solution(mat):
    f = f_all(mat)
    return sum(mu[d] * f[d] for d in range(1, V + 1)) % MOD

for mat, exp in examples:
    print(mobius_solution(mat), "kỳ vọng", exp)

3 kỳ vọng 3
0 kỳ vọng 0


## 8. TRACE từng bước

In [10]:
def trace(mat):
    print("ma trận:", mat)
    f = f_all(mat)
    for d in range(1, 7):
        cs = [multiples_count(r)[d] for r in mat]
        print(f"  d={d}: số phần tử chia hết theo hàng {cs} -> f(d) = {f[d]}, μ(d) = {mu[d]:>2}, đóng góp {mu[d] * f[d]}")
    print("  ... đáp án =", mobius_solution(mat), "| brute =", brute(mat))

trace([[1, 2], [3, 4]])

ma trận: [[1, 2], [3, 4]]
  d=1: số phần tử chia hết theo hàng [2, 2] -> f(d) = 4, μ(d) =  1, đóng góp 4
  d=2: số phần tử chia hết theo hàng [1, 1] -> f(d) = 1, μ(d) = -1, đóng góp -1
  d=3: số phần tử chia hết theo hàng [0, 1] -> f(d) = 0, μ(d) = -1, đóng góp 0
  d=4: số phần tử chia hết theo hàng [0, 1] -> f(d) = 0, μ(d) =  0, đóng góp 0
  d=5: số phần tử chia hết theo hàng [0, 0] -> f(d) = 0, μ(d) = -1, đóng góp 0
  d=6: số phần tử chia hết theo hàng [0, 0] -> f(d) = 0, μ(d) =  1, đóng góp 0
  ... đáp án = 3 | brute = 3


In [11]:
def trace_dp(mat):
    dp = {}
    for x in mat[0]: dp[x] = dp.get(x, 0) + 1
    print("sau hàng 0:", dict(sorted(dp.items())))
    for i, row in enumerate(mat[1:], 1):
        new = {}
        for g, c in dp.items():
            for x in row:
                h = gcd(g, x); new[h] = new.get(h, 0) + c
        dp = new
        print(f"sau hàng {i}:", dict(sorted(dp.items())))
    print("dp[1] =", dp.get(1, 0))

trace_dp([[6, 10, 15], [4, 9, 25], [30, 7]])

sau hàng 0: {6: 1, 10: 1, 15: 1}
sau hàng 1: {1: 3, 2: 2, 3: 2, 5: 2}
sau hàng 2: {1: 12, 2: 2, 3: 2, 5: 2}
dp[1] = 12


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def countCoprime(self, mat: List[List[int]]) -> int:
        MOD = 10**9 + 7
        V = max(max(r) for r in mat)
        mu = [1] * (V + 1)
        is_comp = [False] * (V + 1)
        primes = []
        for i in range(2, V + 1):
            if not is_comp[i]:
                primes.append(i)
                mu[i] = -1
            for p in primes:
                if i * p > V:
                    break
                is_comp[i * p] = True
                if i % p == 0:
                    mu[i * p] = 0
                    break
                mu[i * p] = -mu[i]
        f = [1] * (V + 1)
        for row in mat:
            cnt = [0] * (V + 1)
            for x in row:
                cnt[x] += 1
            for d in range(1, V + 1):
                if mu[d]:
                    c = 0
                    for k in range(d, V + 1, d):
                        c += cnt[k]
                    f[d] = f[d] * c % MOD
        return sum(mu[d] * f[d] for d in range(1, V + 1)) % MOD

sol = Solution()
for mat, exp in examples:
    got = sol.countCoprime(mat)
    print(mat, "->", got, "✔" if got == exp else "✘")

[[1, 2], [3, 4]] -> 3 ✔
[[2, 2], [2, 2]] -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **`f(d)`:** bộ có `d | gcd` ⇔ mỗi phần tử chia hết cho `d` ⇔ mỗi hàng chọn một trong `c_i(d)` phần tử ⇒ `f(d) = Π c_i(d)`.
2. **Möbius:** `[g = 1] = Σ_{d | g} μ(d)`. Lấy tổng trên mọi bộ và đổi thứ tự: `Σ_bộ Σ_{d | gcd(bộ)} μ(d) = Σ_d μ(d) · #{bộ : d | gcd(bộ)} = Σ_d μ(d) f(d)`. Chỉ cần `d ≤ V` vì gcd `≤ V`.
3. **Trừ dần (Cách 3):** `f(d) = Σ_{k ≥ 1} h(k·d)` ⇒ `h(d) = f(d) − Σ_{k≥2} h(kd)`; duyệt `d` giảm dần thì các `h(kd)` đã có.
4. **DP (Cách 2):** bất biến `dp[g]` = số cách chọn các hàng đã xét với gcd đúng `g`; gộp hàng mới theo 3.1.
5. Lời giải nộp bỏ qua `d` có `μ(d) = 0` (không ảnh hưởng tổng).

## 11. Phân tích Time / Space Complexity

Với `V = 150`:

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n^m · m)` | `O(m)` |
| 2. DP gcd | `O(m · V · D)` với `D` số giá trị khác nhau mỗi hàng | `O(V)` |
| 3. Trừ dần | `O(m (n + V log V) + V log V)` | `O(V)` |
| 4. Möbius | `O(m (n + V log V))` | `O(V)` |

## 12. Bộ test

In [13]:
METHODS = [brute, dp_gcd, exact_by_subtraction, mobius_solution, lambda m: Solution().countCoprime(m)]
for mat, exp in examples:
    assert all(f(mat) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên
assert Solution().countCoprime([[1]]) == 1
assert Solution().countCoprime([[2]]) == 0
assert Solution().countCoprime([[7, 1, 14]]) == 1                  # một hàng: chỉ số 1
assert Solution().countCoprime([[1] * 150 for _ in range(150)]) == pow(150, 150, MOD)
print("Ca biên: OK")

Ca biên: OK


In [15]:
# 12.3 ngẫu nhiên
rng = random.Random(3725)
for _ in range(500):
    m = rng.randint(1, 4); n = rng.randint(1, 4)
    mat = [[rng.choice([rng.randint(1, 12), rng.choice([2, 4, 6, 8, 12, 3, 9])]) for _ in range(n)] for _ in range(m)]
    b = brute(mat)
    assert all(f(mat) == b for f in METHODS[1:]), mat
print("500 test ngẫu nhiên: OK")

500 test ngẫu nhiên: OK


In [16]:
# 12.4 test lớn 150 x 150
rng = random.Random(1)
mat = [[rng.randint(1, 150) for _ in range(150)] for _ in range(150)]
t0 = time.time(); x = Solution().countCoprime(mat); t1 = time.time()
y = dp_gcd(mat); t2 = time.time()
print("đáp án:", x, "| Möbius:", round(t1 - t0, 3), "s | DP gcd:", round(t2 - t1, 2), "s | khớp:", x == y)

đáp án: 841474419 | Möbius: 0.006 s | DP gcd: 0.06 s | khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn theo **giá trị phân biệt** thay vì theo ô

Hai ô cùng giá trị là hai lựa chọn khác nhau (ví dụ 2 có đáp án 0 nhưng với `[[1,1],[2]]` đáp án là 2).

In [17]:
def wrong_distinct(mat):
    return brute([sorted(set(r)) for r in mat])

print("sai:", wrong_distinct([[1, 1], [2]]), "| đúng:", brute([[1, 1], [2]]))

sai: 1 | đúng: 2


### Lỗi 2 — Bao hàm – loại trừ chỉ trên **số nguyên tố** đơn lẻ (quên các tích)

In [18]:
def wrong_primes_only(mat):
    f = f_all(mat)
    primes = [p for p in range(2, V + 1) if mu[p] == -1 and all(p % q for q in range(2, p))]
    return (f[1] - sum(f[p] for p in primes)) % MOD                     # SAI: thiếu f(6), f(10), ...

mat = [[6, 1], [6, 5]]
print("sai:", wrong_primes_only(mat), "| đúng:", brute(mat))

sai: 2 | đúng: 3


### Lỗi 3 — Đếm `c_i(d)` bằng "giá trị bằng `d`" thay vì "chia hết cho `d`"

In [19]:
def wrong_equal_not_divisible(mat):
    f = [1] * (V + 1)
    for row in mat:
        for d in range(1, V + 1):
            f[d] = f[d] * (row.count(d) if d > 1 else len(row)) % MOD           # SAI
    return sum(mu[d] * f[d] for d in range(1, V + 1)) % MOD

print("sai:", wrong_equal_not_divisible([[2, 4], [4, 3]]), "| đúng:", brute([[2, 4], [4, 3]]))

sai: 4 | đúng: 2


### Lỗi 4 — Kết quả âm do quên `% MOD` sau tổng có dấu

Trong Python `(-5) % MOD` cho số dương, nhưng nếu trả về tổng **trước** khi lấy modulo, đáp án có thể âm (hoặc rất lớn).

In [20]:
rng0 = random.Random(0)
mat = [[rng0.randint(1, 150) for _ in range(20)] for _ in range(20)]
raw = sum(mu[d] * f_all(mat)[d] for d in range(1, V + 1))          # f(d) đã bị lấy modulo
print("tổng thô:", raw, "| sau % MOD:", raw % MOD, "| DP gcd:", dp_gcd(mat))
assert not (0 <= raw < MOD) and raw % MOD == dp_gcd(mat)

tổng thô: -583352662 | sau % MOD: 416647345 | DP gcd: 416647345


### Lỗi 5 — Khởi tạo DP gcd bằng `dp[0] = 1` rồi gộp mọi hàng

Ý tưởng "gcd(0, x) = x" thật ra **đúng** trong Python (`gcd(0, x) == x`), nên đây là một cách khởi tạo gọn. Lỗi xảy ra khi ai đó khởi tạo `dp[1] = 1` (nhầm "phần tử trung hoà" của gcd là 1): khi đó mọi bộ đều bị kéo về gcd 1.

In [21]:
def wrong_init(mat):
    dp = {1: 1}                                                          # SAI: trung hoà của gcd là 0
    for row in mat:
        new = {}
        for g, c in dp.items():
            for x in row: h = gcd(g, x); new[h] = new.get(h, 0) + c
        dp = new
    return dp.get(1, 0)

print("sai:", wrong_init([[2, 2], [2, 2]]), "| đúng:", brute([[2, 2], [2, 2]]), "| gcd(0, 7) =", gcd(0, 7))

sai: 4 | đúng: 0 | gcd(0, 7) = 7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "gcd của các phần tử được chọn bằng 1" + giá trị nhỏ ⇒ Möbius hoặc trừ dần trên ước.
- "Mọi phần tử chia hết cho `d`" luôn dễ đếm (tích độc lập) ⇒ đi qua `f(d)`.
- Đếm bội: `Σ_{k: d | k} cnt[k]` cho mọi `d` tốn `O(V log V)` (chuỗi điều hoà).
- DP theo gcd hữu ích khi cần **phân phối** gcd hoặc ràng buộc thêm theo thứ tự hàng.
- Khi đáp án là tổng có dấu dưới modulo, luôn lấy `% MOD` **ở cuối** (Python cho kết quả không âm); ở C++/Java cộng thêm `MOD` trước khi lấy dư.
- Sàng Möbius tuyến tính là đoạn mã nên thuộc lòng: cùng khung với sàng số nguyên tố tuyến tính.

## 15. Biến thể của bài

1. **gcd = `G` cho trước:** chia mọi phần tử chia hết cho `G` cho `G`, bỏ phần tử khác, rồi đếm gcd = 1 (hay dùng `h(G)` ở Cách 3).
2. **Đếm số cặp/tập con nguyên tố cùng nhau** (LC 1819-kiểu): `Σ μ(d) (2^{c(d)} − 1)`.
3. **lcm thay cho gcd:** đếm theo ước của lcm — phức tạp hơn.
4. **Tổng gcd của mọi bộ:** `Σ_d φ(d) f(d)` (dùng `Σ_{d|g} φ(d) = g`).

In [22]:
# Biến thể 4: tổng gcd của mọi bộ = Σ φ(d) f(d)
def phi_sieve(N):
    phi = list(range(N + 1))
    for p in range(2, N + 1):
        if phi[p] == p:
            for k in range(p, N + 1, p): phi[k] -= phi[k] // p
    return phi

phi = phi_sieve(V)
def sum_gcd(mat):
    f = f_all(mat)
    return sum(phi[d] * f[d] for d in range(1, V + 1)) % MOD

rng = random.Random(15)
for _ in range(300):
    mat = [[rng.randint(1, 12) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 3))]
    assert sum_gcd(mat) == sum(reduce(gcd, p) for p in product(*mat)) % MOD
print("Biến thể 4 OK; ví dụ 1:", sum_gcd([[1, 2], [3, 4]]))

Biến thể 4 OK; ví dụ 1: 5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Tích Descartes | `O(n^m)` | Kiểm chứng |
| 2. DP gcd | Trạng thái gcd | `O(m V D)` | Cần phân phối gcd |
| 3. Trừ dần | `h(d) = f(d) − Σ h(kd)` | `O(m (n + V log V))` | Không cần μ |
| 4. Möbius | `Σ μ(d) f(d)` | `O(m (n + V log V))` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countCoprime([[2,3],[2,3],[2,3]])`.

**Bài 2 (tính tay).** `countCoprime([[6,10],[15,4]])`.

**Bài 3 (điền TODO).** Viết `count_gcd_equal(mat, G)` = số cách chọn có gcd đúng bằng `G`.

**Bài 4 (tự viết lại).** Tính `μ` bằng phân tích thừa số trực tiếp (không sàng) và so với sàng.

**Bài 5 (mini challenge).** Đếm số cách chọn một số mỗi hàng sao cho gcd **lớn hơn 1**, trong `O(V log V)` sau khi có `f`.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_gcd_equal(mat, G):
    # TODO: h(G) theo Cách 3
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_gcd_gt1(mat):
    # TODO: f(1) - đáp án gcd = 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `f(1) = 8`, `f(2) = 1`, `f(3) = 1`, `f(6) = 0`. (2) `8 − 1 − 1 + 0`. (3) `6`.

**Bài 2.** (1) Các bộ: `(6,15)=3`, `(6,4)=2`, `(10,15)=5`, `(10,4)=2`. (2) Không bộ nào gcd 1. (3) `0`.

**Bài 3.** (1) `h(d)` từ trên xuống. (2) Trả `h(G)`. (3) Kiểm chứng vét cạn.

**Bài 4.** (1) Phân tích `d` bằng chia thử. (2) Có thừa số lặp ⇒ 0. (3) `(−1)^{số thừa số}`.

**Bài 5.** (1) `f(1) = Π n_i` = tổng số bộ. (2) Trừ số bộ gcd = 1. (3) Lấy modulo.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().countCoprime([[2, 3], [2, 3], [2, 3]]) == 6 == brute([[2, 3], [2, 3], [2, 3]])
assert Solution().countCoprime([[6, 10], [15, 4]]) == 0 == brute([[6, 10], [15, 4]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3, 4
def count_gcd_equal_sol(mat, G):
    f = f_all(mat); h = [0] * (V + 1)
    for d in range(V, 0, -1):
        h[d] = (f[d] - sum(h[k] for k in range(2 * d, V + 1, d))) % MOD
    return h[G] if G <= V else 0

def mu_direct(d):
    r = 0; p = 2
    while p * p <= d:
        if d % p == 0:
            d //= p
            if d % p == 0: return 0
            r += 1
        p += 1
    if d > 1: r += 1
    return -1 if r % 2 else 1

assert all(mu_direct(d) == mu[d] for d in range(1, V + 1))
rng = random.Random(4)
for _ in range(300):
    mat = [[rng.randint(1, 12) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 3))]
    G = rng.randint(1, 6)
    assert count_gcd_equal_sol(mat, G) == sum(1 for p in product(*mat) if reduce(gcd, p) == G)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [27]:
# Bài 5
def count_gcd_gt1_sol(mat):
    total = 1
    for r in mat: total = total * len(r) % MOD
    return (total - Solution().countCoprime(mat)) % MOD

assert count_gcd_gt1_sol([[1, 2], [3, 4]]) == 1
assert count_gcd_gt1_sol([[2, 2], [2, 2]]) == 4
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
mu = sàng Möbius tới V
f[d] = Π_hàng #{x : d | x}                 # đếm bội của d trong từng hàng
ans = Σ_{d=1..V} mu[d] * f[d]   (mod 1e9+7)
```

**Checklist:**

- [ ] Đếm theo **ô** (giá trị trùng vẫn tính riêng)?
- [ ] `c_i(d)` là số phần tử **chia hết** cho `d`?
- [ ] Đủ mọi `d` (không chỉ số nguyên tố)?
- [ ] Lấy `% MOD` sau tổng có dấu?
- [ ] Đã so với vét cạn?